## SageMaker PySDK v3: Using SageMaker AI Inference Benchmarking on an LMI/DJL Endpoint with a Custom I/O Formatter

This notebook first deploys the open-weight **`gpt-oss-20b`** model (artifacts already staged in S3) to a SageMaker real-time endpoint using the **SageMaker Python SDK v3**, serving it with the latest **LMI/DJL** container behind a custom input/output formatter (`model.py`) that handles request and response shaping. With the endpoint live, it then turns to the main focus: benchmarking.

Using **SageMaker AI Inference Benchmarking**, the notebook drives load against the endpoint and reports its performance characteristics — latency, throughput, and concurrency behavior — so you can right-size the deployment before putting it into production. Because the endpoint uses a custom I/O formatter rather than a standard JumpStart or Hugging Face Hub signature, the benchmark is configured with sample payloads that match the formatter's expected request and response shapes, ensuring the measured numbers reflect the real serving path.


For the `ModelBuilder` + AI Inference Recommender workflow against JumpStart models, see [`pysdk-ai-inference-recommender-demo.ipynb`](pysdk-ai-inference-recommender-demo.ipynb).

Steps:

1. **Setup** — Install the official `sagemaker` PyPI package, resolve IAM role
2. **Build** — Write custom input/output formatters (`model.py`) and `serving.properties`, package artifacts to S3
3. **Deploy** — Use `sagemaker.core.resources` to create Model, EndpointConfig, Endpoint served with latest LMI/DJL container
4. **Inference** — Invoke the endpoint via python SDK v3 to demonstrate working of custom input and output formatter endpoint
5. **Benchmark** — Use `start_benchmark` with `Workload.template(...)` (custom Jinja2 payload template) since the endpoint doesn't speak the OpenAI-chat format
6. **Cleanup** — Delete all resources

**Model:** gpt-oss-20b (pre-uploaded to S3)  
**Instance:** ml.g6e.2xlarge (NVIDIA L40S GPU)  
**Container:** Latest DJL LMI container (resolved dynamically via SDK)  

**Prerequisites:**
- IAM role with SageMaker, ECR, and S3 access
- Service quota for `ml.g7e.2xlarge` or `ml.g6e.2xlarge` endpoint instances
- The SageMaker Python SDK v3 (`pip install sagemaker`), which pulls in `sagemaker-core`, `sagemaker-serve`, and `sagemaker-train`

## 1. Setup and Dependencies

In [ ]:
pip install --upgrade -q "sagemaker>=3.16.0"

In [ ]:
import os
import json
import time
import uuid
import logging
import sys

# Stream INFO logs so progress is visible in cell output.
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                    stream=sys.stdout, force=True)
log = logging.getLogger("demo")

In [ ]:
# Shared config
INSTANCE_TYPE = "ml.g6e.2xlarge"

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Resolve region from the SageMaker session — works correctly inside Studio/notebooks.
# Does NOT fall back to a hardcoded region; raises clearly if region can't be determined.
_sm_session = Session()
REGION = _sm_session.boto_session.region_name
if not REGION:
    raise RuntimeError(
        "Could not determine AWS region. Set the AWS_DEFAULT_REGION environment variable "
        "or run this notebook inside SageMaker Studio."
    )

account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
ROLE = get_execution_role(sagemaker_session=_sm_session)

# MODEL_S3_URI points to where the download cell uploads the model.
# If your model is already in S3 at a different path, update this accordingly.
MODEL_S3_URI = f"s3://sagemaker-{REGION}-{account_id}/lmi-models/gpt-oss-20b/"

# Unique identifiers for this run
import time, uuid
uid = f"{int(time.time())}-{uuid.uuid4().hex[:8]}"
model_name = f"gpt-oss-20b-pysdk-{uid}"
endpoint_name = f"gpt-oss-20b-pysdk-ep-{uid}"

print(f"Region:        {REGION}")
print(f"Account:       {account_id}")
print(f"Role:          {ROLE}")
print(f"Instance type: {INSTANCE_TYPE}")
print(f"Model S3 URI:  {MODEL_S3_URI}")
print(f"Model name:    {model_name}")
print(f"Endpoint name: {endpoint_name}")

## 1b. Download Model and Upload to S3

If you don't have the model pre-uploaded to S3, run this section to:
1. Download `gpt-oss-20b` from Hugging Face Hub using `huggingface_hub`
2. Upload the model files to S3 in your current region
3. Update `MODEL_S3_URI` so the rest of the notebook uses the correct location

**Skip this section** if your model is already in S3.

In [ ]:
import subprocess, sys, os, pathlib, boto3
from datetime import datetime

# ── Configure ────────────────────────────────────────────────────────────────
HF_MODEL_ID = "openai/gpt-oss-20b"
HF_TOKEN    = None                            # Set to "hf_xxxx" if the repo is gated

LOCAL_DIR   = pathlib.Path("./downloaded_model")

account_id   = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
s3_bucket    = f"sagemaker-{REGION}-{account_id}"
s3_prefix    = "lmi-models/gpt-oss-20b"
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: Install huggingface_hub with fast transfer support
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "huggingface_hub[hf_transfer]"])
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# Suppress widget-based progress bars — works across all huggingface_hub versions
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Step 2: Download from Hugging Face
from huggingface_hub import snapshot_download

print(f"Downloading {HF_MODEL_ID} → {LOCAL_DIR} ...")
snapshot_download(
    repo_id=HF_MODEL_ID,
    local_dir=str(LOCAL_DIR),
    token=HF_TOKEN,
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*",
                     "rust_model*", "*.ot", "original/"],
)
print(f"Download complete. Files: {sum(1 for _ in LOCAL_DIR.rglob('*') if _.is_file())}")

# Step 3: Upload to S3
s3_client = boto3.client("s3", region_name=REGION)

try:
    s3_client.head_bucket(Bucket=s3_bucket)
except s3_client.exceptions.ClientError:
    if REGION == "us-east-1":
        s3_client.create_bucket(Bucket=s3_bucket)
    else:
        s3_client.create_bucket(
            Bucket=s3_bucket,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )
    print(f"Created bucket: {s3_bucket}")

all_files = [f for f in LOCAL_DIR.rglob("*") if f.is_file()]
print(f"\nUploading {len(all_files)} files to s3://{s3_bucket}/{s3_prefix}/ ...")

for i, local_file in enumerate(all_files, 1):
    relative = local_file.relative_to(LOCAL_DIR)
    s3_key = f"{s3_prefix}/{relative}"
    s3_client.upload_file(str(local_file), s3_bucket, s3_key)
    if i % 10 == 0 or i == len(all_files):
        print(f"  {i}/{len(all_files)} uploaded")

# Step 4: Update MODEL_S3_URI for the rest of the notebook
MODEL_S3_URI = f"s3://{s3_bucket}/{s3_prefix}/"
print(f"\n✅ Model uploaded. Updating MODEL_S3_URI = {MODEL_S3_URI}")

import shutil
shutil.rmtree(LOCAL_DIR)
print("Local copy removed.")

## 2. Define Custom Input and Output Formatters

The DJL/LMI container supports custom input and output formatters via a `model.py` file
using the `@input_formatter` and `@output_formatter` decorators.

**Input Formatter** (`@input_formatter`):
- Receives the raw Input object, decodes it, transforms the payload
- Supports `{"inputs": ...}`, `{"prompt": ...}`, and `{"messages": [...]}` formats

**Output Formatter** (`@output_formatter`):
- Receives token-by-token output and produces a single JSON response on the last token
- Adds metadata (timestamp, engine, formatter version) and usage stats

**Critical:** Do NOT include a `handle()` function — the container auto-discovers decorators.

In [ ]:
import os

# Create the model.py with decorator-based custom input and output formatters
model_py_content = '''import time
import logging
from djl_python.input_parser import input_formatter
from djl_python.output_formatter import output_formatter
from djl_python.encode_decode import decode
from djl_python.request_io import TextInput

logger = logging.getLogger(__name__)


@input_formatter
def custom_input_formatter(input_item, **kwargs):
    """
    Custom input formatter for DJL 0.36.0 / LMI v25.
    Supports: {"inputs": ...}, {"prompt": ...}, {"messages": [...]}
    """
    content_type = input_item.get_property("Content-Type")
    decoded_payload = decode(input_item, content_type)

    logger.info(f"Input formatter received payload keys: {list(decoded_payload.keys())}")

    # Flexible input field mapping
    if "prompt" in decoded_payload and "inputs" not in decoded_payload:
        decoded_payload["inputs"] = decoded_payload.pop("prompt")

    # Chat messages support
    tokenizer = kwargs.get("tokenizer", None)
    if "messages" in decoded_payload and tokenizer is not None:
        messages = decoded_payload["messages"]
        if not any(m.get("role") == "system" for m in messages):
            messages.insert(0, {
                "role": "system",
                "content": "You are a helpful, accurate, and concise AI assistant."
            })
        try:
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            decoded_payload["inputs"] = prompt
            decoded_payload.pop("messages", None)
        except Exception as e:
            logger.warning(f"Chat template failed: {e}. Falling back to concat.")
            prompt_parts = []
            for msg in messages:
                role = msg.get("role", "user")
                content = msg.get("content", "")
                prompt_parts.append(f"<|{role}|>\\n{content}")
            prompt_parts.append("<|assistant|>\\n")
            decoded_payload["inputs"] = "\\n".join(prompt_parts)
            decoded_payload.pop("messages", None)

    # Extract inputs and parameters
    input_text = decoded_payload.pop("inputs", "")
    params = decoded_payload.pop("parameters", {})

    # Set sensible defaults
    params.setdefault("max_new_tokens", 256)
    params.setdefault("temperature", 0.7)
    params.setdefault("top_p", 0.9)
    params["max_new_tokens"] = min(int(params["max_new_tokens"]), 2048)
    params["temperature"] = max(0.0, min(float(params["temperature"]), 2.0))

    # Build and return TextInput
    request_input = TextInput()
    request_input.input_text = input_text
    request_input.parameters = params
    request_input.tokenizer = tokenizer
    return request_input


@output_formatter
def custom_output_formatter(token, first_token, last_token, details, generated_tokens, request_output, **kwargs):
    """
    Custom output formatter for DJL 0.36.0 / LMI v25 or above.
    Emits a single JSON response on the last token.
    """
    import json

    if not last_token:
        return ""

    full_text = generated_tokens + token.text
    result = {
        "generated_text": full_text,
        "metadata": {
            "timestamp": int(time.time()),
            "formatter_version": "1.0.0",
            "engine": "vllm",
        },
        "usage": {
            "generated_chars": len(full_text),
            "generated_words": len(full_text.split()),
        },
        "finish_reason": details.get("finish_reason", "unknown") if details else "unknown",
    }
    return json.dumps(result)
'''

# Write model.py
os.makedirs("model_artifacts", exist_ok=True)
with open("model_artifacts/model.py", "w") as f:
    f.write(model_py_content)

print("Created model_artifacts/model.py")

In [ ]:
# Create serving.properties
# NOTE: No engine= line! The container auto-selects based on option.rolling_batch
serving_properties_content = """option.model_id={model_s3}
option.rolling_batch=vllm
option.tensor_parallel_degree=1
option.max_rolling_batch_size=32
option.dtype=auto
option.max_model_len=4096
option.trust_remote_code=true
""".format(model_s3=MODEL_S3_URI)

with open("model_artifacts/serving.properties", "w") as f:
    f.write(serving_properties_content)

print("Created model_artifacts/serving.properties")
print(serving_properties_content)

In [ ]:
# Upload model artifacts (uncompressed) to S3
# LLMs should NOT be packaged as tarballs — keep artifacts uncompressed for faster loading.
import boto3
from datetime import datetime

account_id = boto3.client("sts").get_caller_identity()["Account"]
default_bucket = f"sagemaker-{REGION}-{account_id}"

timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
s3_prefix = f"lmi-models/gpt-oss-20b-pysdk/{timestamp}"

s3_client = boto3.client("s3", region_name=REGION)
for filename in ["model.py", "serving.properties"]:
    s3_client.upload_file(f"model_artifacts/{filename}", default_bucket, f"{s3_prefix}/{filename}")

s3_artifact_uri = f"s3://{default_bucket}/{s3_prefix}/"
print(f"Uploaded uncompressed artifacts to: {s3_artifact_uri}")

## 3. Build and Deploy with sagemaker-core Resources

Since `gpt-oss-20b` is a fully custom model (not on JumpStart or HuggingFace Hub),
we use the `sagemaker.core.resources` classes (`Model`, `EndpointConfig`, `Endpoint`)
from the new PySDK to create and deploy. This replaces the raw boto3 calls while
still giving us full control over the container image and model artifacts.

After deployment, we use `start_benchmark` from `sagemaker.serve` to measure performance.

In [ ]:
from sagemaker.core.resources import Model, EndpointConfig, Endpoint

# Retrieve the latest LMI container image URI dynamically via the SDK
from sagemaker.core import image_uris
image_uri = image_uris.retrieve(framework="djl-lmi", region=REGION, version="latest")
print(f"Using LMI container: {image_uri}")

# Create the SageMaker Model using sagemaker-core
core_model = Model.create(
    model_name=model_name,
    execution_role_arn=ROLE,
    primary_container={
        "image": image_uri,
        "model_data_source": {
            "s3_data_source": {
                "s3_uri": s3_artifact_uri,   # must end with a trailing slash
                "s3_data_type": "S3Prefix",
                "compression_type": "None",
            }
        },
    },
)

print(f"Model created: {core_model.model_name}")

In [ ]:
# Recovery cell: run this if core_endpoint is not defined after a kernel restart.
# It reconnects to an existing endpoint rather than re-deploying.
from sagemaker.core.resources import Endpoint, EndpointConfig

import boto3
sm_client = boto3.client("sagemaker", region_name=REGION)

try:
    core_endpoint
    print(f"core_endpoint already defined: {core_endpoint.endpoint_name}")
except NameError:
    # Try to find an existing InService endpoint matching the current uid
    try:
        desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
        status = desc["EndpointStatus"]
        if status == "InService":
            core_endpoint = Endpoint.get(endpoint_name=endpoint_name)
            endpoint_config_name = f"gpt-oss-20b-pysdk-epc-{uid}"
            core_endpoint_config = EndpointConfig.get(endpoint_config_name=endpoint_config_name)
            print(f"Reconnected to existing endpoint: {endpoint_name} ({status})")
        else:
            print(f"Endpoint exists but status is '{status}' — re-run the deploy cell below.")
    except sm_client.exceptions.ClientError:
        print(f"No existing endpoint found for uid={uid}. Run the deploy cell below to create one.")

In [ ]:
# Deploy the model to an endpoint
log.info("Deploying endpoint (this may take 10-15 minutes for model download and loading)...")

# Create Endpoint Config
endpoint_config_name = f"gpt-oss-20b-pysdk-epc-{uid}"

core_endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        {
            "variant_name": "AllTraffic",
            "model_name": model_name,
            "instance_type": INSTANCE_TYPE,
            "initial_instance_count": 1,
            "container_startup_health_check_timeout_in_seconds": 900,
        }
    ],
)
print(f"Endpoint config created: {endpoint_config_name}")

# Create and wait for the Endpoint
core_endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)

core_endpoint.wait_for_status("InService")
print(f"Endpoint InService: {core_endpoint.endpoint_name}")

## 4. Run Inference

The custom input formatter accepts:
- `"inputs"` field (standard LMI format)
- `"prompt"` field (mapped to `inputs` by the formatter)
- `"messages"` field (OpenAI chat format, converted via tokenizer chat template)

The custom output formatter enriches responses with:
- `metadata` — timestamp, formatter version, engine
- `usage` — character count, word count
- `finish_reason` — promoted to top level

In [ ]:
import json

from sagemaker.core.resources import Endpoint

# Reconnect to the endpoint resource if this cell runs after a kernel restart.
if "core_endpoint" not in globals():
    core_endpoint = Endpoint.get(endpoint_name=endpoint_name)
    print(f"Reconnected to endpoint: {core_endpoint.endpoint_name}")


def invoke_json(payload):
    """Invoke the endpoint via the PySDK v3 Endpoint resource and parse the JSON body."""
    response = core_endpoint.invoke(
        body=json.dumps(payload),
        content_type="application/json",
        accept="application/json",
    )
    return json.loads(response.body.read().decode("utf-8"))


# Example 1: Standard "inputs" field
payload = {
    "inputs": "The future of artificial intelligence is",
    "parameters": {
        "max_new_tokens": 128,
        "temperature": 0.8,
    }
}

result = invoke_json(payload)
print("Generated text:")
print(result.get("generated_text", ""))
print(f"\nFinish reason: {result.get('finish_reason', 'N/A')}")
print(f"\nMetadata: {json.dumps(result.get('metadata', {}), indent=2)}")
print(f"\nUsage: {json.dumps(result.get('usage', {}), indent=2)}")

## 5. Benchmark the Endpoint

Use the new `start_benchmark` API to measure endpoint performance with a synthetic workload.
This runs AIPerf against the deployed endpoint and returns structured metrics
(throughput, latency, TTFT, ITL, etc.).

In [ ]:
# ── Recovery cell ────────────────────────────────────────────────────────────
# Run this if you restarted the kernel and want to continue from an existing
# endpoint without re-deploying. Skip if you just ran the deploy cell above.

from sagemaker.core.resources import Endpoint
from sagemaker.core.helper.session_helper import Session, get_execution_role
import boto3

if "REGION" not in globals():
    REGION = Session().boto_session.region_name
if "ROLE" not in globals():
    ROLE = get_execution_role(sagemaker_session=Session())

account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
default_bucket = f"sagemaker-{REGION}-{account_id}"

endpoint_name = globals().get("endpoint_name") or input("Enter the existing endpoint name to reconnect to: ").strip()

if "core_endpoint" not in globals():
    core_endpoint = Endpoint.get(endpoint_name=endpoint_name)
    print(f"Reconnected: {endpoint_name} ({core_endpoint.endpoint_status})")
else:
    print(f"core_endpoint already set: {core_endpoint.endpoint_name}")
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Resolve ROLE/REGION in case this cell runs without the prior setup cells.
if "REGION" not in globals():
    REGION = Session().boto_session.region_name
    if not REGION:
        raise RuntimeError("Could not determine AWS region. Run the shared config cell first.")

if "ROLE" not in globals():
    ROLE = get_execution_role(sagemaker_session=Session())

# --- Step 1: Define the Jinja2 template + response field for the benchmark ---
# The custom input formatter accepts {"inputs": "...", "parameters": {...}, "stream": bool}.
# Workload.template() (next cell) uploads this template to S3 for you — no manual put_object.
TEMPLATE_CONTENT = '{\n  "inputs": {{ text|tojson }},\n  "parameters": {\n    "max_new_tokens": {{ max_tokens }},\n    "temperature": 0.7,\n    "top_p": 0.9\n  },\n  "stream": {{ stream|tojson }}\n}'

# JMESPath to extract the generated text from our custom output formatter's response.
response_field = "generated_text"

print(f"Template content:\n{TEMPLATE_CONTENT}")

In [ ]:
# --- Step 2: Run the benchmark with template mode ---
# Build the workload with Workload.template — it sets payload_template +
# response_field for you and wires the template into the workload's
# dataset_config. Then pass it to start_benchmark via workload=.
from sagemaker.serve import start_benchmark, Workload

workload = Workload.template(
    request_template=TEMPLATE_CONTENT,   # inline Jinja2 string, OR a local file path — SDK uploads it for you
    response_field=response_field,        # e.g. "generated_text"
    tokenizer="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    concurrency=4,
    request_count=100,
    prompt_input_tokens_mean=100,
    output_tokens_mean=50,
    streaming=False,
)

job = start_benchmark(
    endpoint=core_endpoint,
    workload=workload,     # pass the workload object — do NOT pass dataset_config / extra_inputs to start_benchmark
    role=ROLE,
    wait=True,
)

print(f"Benchmark terminal state: {job.ai_benchmark_job_status}")

# Parse and display results
result = job.show_result()
print(result)

# Pick specific metrics
print(f"\n--- Key Metrics ---")
print(f"Request throughput avg: {result.metrics.request_throughput.avg} req/sec")
print(f"Request latency p99:   {result.metrics.request_latency.p99} ms")

ott = result.metrics.get("output_token_throughput")
if ott:
    print(f"Output token throughput avg: {ott.avg} {ott.unit}")

In [ ]:
# Pick specific metrics from the benchmark result.
# Guard against running this cell standalone / out of order — if `job`
# isn't in scope (e.g. kernel was restarted), look up the most recent
# benchmark job for this endpoint instead of failing outright.
if "job" not in dir():
    from sagemaker.core.resources import AIBenchmarkJob

    if "core_endpoint" not in dir():
        raise NameError(
            "Neither `job` nor `core_endpoint` is defined. Run the setup "
            "and benchmark cells above first."
        )

    matches = sorted(
        (
            j
            for j in AIBenchmarkJob.get_all()
            if getattr(getattr(j, "benchmark_target", None), "endpoint", None)
            and j.benchmark_target.endpoint.identifier == core_endpoint.endpoint_name
        ),
        key=lambda j: j.creation_time,
        reverse=True,
    )
    if not matches:
        raise NameError(
            f"No AIBenchmarkJob found for endpoint '{core_endpoint.endpoint_name}'. "
            "Run the benchmark cell above first (the one that calls start_benchmark(...))."
        )
    job = matches[0]

if "result" not in dir():
    result = job.show_result()

# Well-known shortcuts (typed; IDE autocomplete works):
print(f"Request throughput avg: {result.metrics.request_throughput.avg} req/sec")
print(f"E2E latency p90:       {result.metrics.request_latency.p90} ms")

# Any metric AIPerf produced, by raw key:
itl = result.metrics.get("inter_token_latency")
if itl:
    print(f"Inter-token latency p50: {itl.p50} {itl.unit}")

ott = result.metrics.get("output_token_throughput")
if ott:
    print(f"Output token throughput avg: {ott.avg} {ott.unit}")

## 6. Cleanup

Delete the endpoint, endpoint config, model, and benchmark job to avoid ongoing charges.

In [ ]:
from sagemaker.core.resources import AIBenchmarkJob, AIWorkloadConfig, EndpointConfig

def _try(label, fn):
    # fn is called (not evaluated) inside the try block, so a NameError
    # raised by referencing an undefined variable (e.g. this cell was run
    # after a kernel restart without re-running earlier cells) is caught
    # here instead of crashing the whole cleanup cell.
    try:
        fn()
        print(f"  Deleted: {label}")
    except NameError as e:
        print(f"  [skip] {label}: not defined in this session ({e})")
    except Exception as e:
        print(f"  [skip] {label}: {e}")

# Delete endpoint, endpoint config, model
# Use lambdas so the variable lookup happens inside _try's try/except
# rather than while building the argument list (which would raise
# NameError immediately and abort the whole cell).
_try("endpoint", lambda: core_endpoint.delete())
_try("endpoint config", lambda: core_endpoint_config.delete())
_try("model", lambda: core_model.delete())

# Delete benchmark job + workload config
_try("benchmark job", lambda: job.delete())
_try("workload config",
     lambda: AIWorkloadConfig.get(
         ai_workload_config_name=job.ai_workload_config_identifier
     ).delete())

print("\nDone! All resources cleaned up.")

In [ ]:
# Clean up local artifacts
import shutil

if os.path.exists("model_artifacts"):
    shutil.rmtree("model_artifacts")

print("Local artifacts cleaned up.")